In [ ]:
""" Flatten_json_wide, nested objects become columns and arrays become indexed columns."""

import base64, json, logging, os, re
from datetime import datetime
from pathlib import Path
from typing import Any
import pandas as pd

logging.basicConfig(
    level=os.getenv("LOG_LEVEL", "INFO"),
    format="%(asctime)s %(levelname)s [%(name)s] %(message)s",
    force=True,
)
logger = logging.getLogger(__name__)

LAKEHOUSE_ROOT = os.getenv("LAKEHOUSE_ROOT", "/lakehouse/default")
LANDING_ROOT = Path(LAKEHOUSE_ROOT) / "Files" / "landing"

UNWANTED_CHARS = set('#@!$%^&*()-+={}[]|\\:;"\'<>,.?/~` ')


def decode_config_json(param_string_b64: str) -> dict[str, Any]:
    """Decode Airflow/Fabric parameter. Accepts URL-safe base64, standard base64, or plain JSON."""
    payload = (param_string_b64 or "").strip()
    if not payload:
        raise ValueError("Parameter 'param_string_b64' is empty.")

    for decoder in (base64.urlsafe_b64decode, base64.b64decode):
        try:
            decoded = decoder(payload.encode("utf-8")).decode("utf-8")
            return json.loads(decoded)
        except Exception:
            pass

    try:
        return json.loads(payload)
    except json.JSONDecodeError as exc:
        raise ValueError("param_string_b64 must contain valid JSON config, base64 encoded or plain JSON.") from exc


def is_activated(value: Any) -> bool:
    return str(value).strip().lower() == "true" if not isinstance(value, bool) else value


def clean_column_name(name: str) -> str:
    cleaned = ''.join('_' if ch in UNWANTED_CHARS else ch for ch in str(name))
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")
    return cleaned or "col"


def make_unique_columns(columns: list[str]) -> list[str]:
    seen: dict[str, int] = {}
    result: list[str] = []
    for col in columns:
        base = clean_column_name(col)
        if base not in seen:
            seen[base] = 0
            result.append(base)
        else:
            seen[base] += 1
            result.append(f"{base}_{seen[base]}")
    return result


def read_json_file(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def records_from_json(data: Any) -> list[Any]:
    if isinstance(data, list):
        return data
    return [data]


def write_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False, engine="pyarrow")


In [ ]:

def flatten_wide_record(value: Any, prefix: str = "") -> dict[str, Any]:
    """Flatten dicts and widen arrays into indexed columns: items_0_sku, items_1_sku, ..."""
    result: dict[str, Any] = {}

    if isinstance(value, dict):
        for key, item in value.items():
            next_prefix = f"{prefix}_{key}" if prefix else str(key)
            result.update(flatten_wide_record(item, next_prefix))
    elif isinstance(value, list):
        if not value:
            result[prefix] = None
        for index, item in enumerate(value):
            next_prefix = f"{prefix}_{index}" if prefix else str(index)
            result.update(flatten_wide_record(item, next_prefix))
    else:
        result[prefix] = value

    return result


def flatten_wide(data: Any) -> pd.DataFrame:
    rows = [flatten_wide_record(record) for record in records_from_json(data)]
    df = pd.DataFrame(rows)
    if not df.empty:
        df.columns = make_unique_columns(list(df.columns))
        df = df.drop_duplicates(ignore_index=True)
    return df


In [ ]:

# Fabric/Airflow injects this parameter at runtime.

try:
    param_string_b64
except NameError:
    param_string_b64 = ""

config = decode_config_json(param_string_b64)
source_system = str(config.get("source_system", "")).strip()
if not source_system:
    raise ValueError("Parameter 'source_system' is required.")
source_system_lower = source_system.lower()

execution_date_str = str(config.get("execution_date", "")).strip()[:10]
if not execution_date_str:
    raise ValueError("Parameter 'execution_date' is required.")
execution_date = datetime.strptime(execution_date_str, "%Y-%m-%d")
year, month, day = execution_date.year, execution_date.month, execution_date.day

sub_source_system = config.get("sub_source_system")
sub_source_system = str(sub_source_system).strip() if sub_source_system else None
lakehouse_base_path = f"{source_system_lower}/{sub_source_system}" if sub_source_system else source_system_lower

tables = [t for t in config.get("tables", []) if is_activated(t.get("activated"))]
if not tables:
    raise ValueError("Parameter 'tables' must contain at least one activated table.")

logger.info("Decoded parameters for source system '%s' on %s with %s activated table(s).", source_system, execution_date_str, len(tables))

for table in tables:
    source_table = str(table.get("source_table", "")).strip()
    if not source_table:
        raise ValueError("Each table entry must include 'source_table'.")

    base_name = f"{source_system}_{source_table}_{year}{month:02d}{day:02d}_data"
    json_path = LANDING_ROOT / lakehouse_base_path / source_table / str(year) / f"{month:02d}" / f"{day:02d}" / f"{base_name}.json"
    parquet_path = LANDING_ROOT / lakehouse_base_path / source_table / str(year) / f"{month:02d}" / f"{day:02d}" / f"{base_name}.parquet"

    logger.info("Processing table '%s'", source_table)
    logger.info("Reading JSON input from %s", json_path)

    if not json_path.exists():
        logger.warning("JSON file not found: %s. Skipping table.", json_path)
        continue

    data = read_json_file(json_path)
    df = flatten_wide(data)

    if df.empty:
        logger.warning("Source JSON for table '%s' is empty. Skipping parquet write.", source_table)
        continue

    logger.info("Writing %s rows and %s columns to %s", len(df), len(df.columns), parquet_path)
    write_parquet(df, parquet_path)
    logger.info("Completed parquet write for table '%s'", source_table)
